# CBD Family Comparison

Compares CBD, M6, M7, M8 models on AIC/BIC and parameter visualisation.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pystmomo as ps

data = ps.load_ew_male()

## Fit All Models

In [ ]:
models = {
    'LC':  ps.lc(),
    'CBD': ps.cbd(),
    'APC': ps.apc(),
    'RH':  ps.rh(),
    'M6':  ps.m6(),
    'M7':  ps.m7(),
    'M8':  ps.m8(),
}

fits = {}
for name, model in models.items():
    fits[name] = model.fit(data.deaths, data.exposures,
                           ages=data.ages, years=data.years)
    print(f"{name:4s}  deviance={fits[name].deviance:12.1f}  AIC={fits[name].aic():12.1f}")

## AIC/BIC Comparison

In [ ]:
summary = pd.DataFrame({
    name: {'Deviance': f.deviance, 'AIC': f.aic(), 'BIC': f.bic(), 'npar': f.npar}
    for name, f in fits.items()
}).T.astype(float).round(1)
print(summary.sort_values('AIC'))

## CBD Parameters

In [ ]:
ps.plot_parameters(fits['CBD'])
plt.suptitle('CBD Model Parameters')
plt.tight_layout()
plt.show()

## M7 Parameters (includes cohort effect)

In [ ]:
ps.plot_parameters(fits['M7'])
plt.suptitle('M7 Model Parameters')
plt.tight_layout()
plt.show()

## Forecast Comparison at Age 65

In [ ]:
age_idx = np.where(data.ages == 65)[0][0]
fig, ax = plt.subplots(figsize=(12, 5))

# Observed
obs_rates = data.deaths[age_idx] / data.exposures[age_idx]
ax.semilogy(data.years, obs_rates, 'k.', ms=4, label='Observed', zorder=10)

colors = plt.cm.tab10(np.linspace(0, 0.7, len(fits)))
for (name, fit), color in zip(fits.items(), colors):
    fc = ps.forecast(fit, h=20)
    years_all = list(data.years) + list(fc.years_f)
    rates_all = list(fit.fitted_rates[age_idx]) + list(fc.rates[age_idx])
    ax.semilogy(years_all, rates_all, color=color, label=name, lw=1.5)

ax.axvline(data.years[-1], color='gray', ls='--', alpha=0.5)
ax.set_xlabel('Year'); ax.set_ylabel('Mortality rate')
ax.set_title('Mortality forecast comparison — Age 65')
ax.legend(ncol=2); plt.tight_layout(); plt.show()